# Genetic Algorithms

## 1. Local Beam Search

In [ ]:
class Action:
    pass

class State:
    def is_goal(self) -> bool:
        pass

    def successor_fn(self) -> list[tuple[Action, State]]:
        pass

    def eval(self) -> float:
        pass

In [8]:
import heapq

class BoundedPriorityQueue:
    def __init__(self, k: int):
        self.arr = []
        self.k = k

    def __len__(self):
        return len(self.arr)

    def empty(self) -> bool:
        return len(self) == 0

    def push(self, element: State, key: float) -> None:
        if len(self) < self.k:
            self.arr.append((key, element))
        elif self.arr[0][0] < key:
            heapq.heappop(self.arr)
            heapq.heappush(self.arr, (key, element))

    def drain(self) -> list[State]:
        arr = [element for _, element in self.arr]
        self.arr = []
        return arr

In [ ]:
def local_beam_search(states: list[State], iterations: int) -> State:
    k = len(states)
    for _ in range(0, iterations):
        candidates = BoundedPriorityQueue(k)
        for state in states:
            for action, next_state in state.successor_fn():
                if next_state.is_goal():
                    return next_state
                candidates.push(next_state, next_state.eval())
        states = candidates.drain()
    return states[0]

## 2. Genetic Algorithms

### 2.1 Definitions

**Genetic algorithms** = **stochastic local beam search** + **generate successors from pairs of states**

Genetic algorithms are:

- **metaheuristic** inspired by process of *natural selection* that belongs to the larger class of
evolutionary algorithms. Heuristics are problem-specific while metaheuristics provide generalized templates.
- commonly used in generation of high-quality solutions to optimization and search problems.
- a type of local search that mimics evolution by working on **populations of strings**,
which encode possible solutions and combines them based on a fitness function to produce individuals that are more fit

In [ ]:
# this is a pseudocode
def genetic_algorithm():
    population = population_initialization()
    scores = fitness_fn(population)
    while is_goal(population, scores):
        children = []
        while does_it_need_more_children(children):
            p1 = parent_selection(population, scores, None)
            p2 = parent_selection(population, scores, p1)
            child = crossover(p1, p2)
            child = mutation(child)
            children.append(child)
        population = population_generation(population, scores, children)
        scores = fitness_fn(population)
    return population, scores

The main steps are:

- initialization
- fitness function
- **selection**
- **crossover**
- **mutation**

### 2.2 Initialization. Population. Individual

Similar to local beam search, **GAs** (Genetic Algorithms) start with **k randomly generated states**, called the
**population**. Each state (also called **individual**) is represented as string over a finite alphabet.

### 2.3 Fitness Function

The **fitness function** is just GA terminology for evaluation function,
which returns higher values for better states. For example in N-queens problem, this can be the number of nonattacking pairs of queens.

### 2.4 Selection

Selection is the stage of a genetic algorithm in which individual genomes are chosen from a population for
later breeding (using the crossover operator).

Selection methods:

- Roulette Wheel Selection (Fitness Proportionate Selection)
- Tournament Selection
- Rank-Based Selection
- Stochastic Universal Sampling (SUS)
- Elitism
- Truncation Selection

In [112]:
population = ["A", "B", "C", "D"]
scores = [60, 5, 30, 5]

In [70]:
import random

def roulette_wheel_selection(population: list[str], scores: list[float]) -> str:
    """
    Divides the interval [0, 1) to slices, proportionate to the scores.
    Scores must be non-negative.
    One very fit individual nukes the diversity.
    If all scores are almost the same, this selection behaves like random selection.
    """
    total_score = sum(scores)
    spin = random.uniform(0, total_score)
    accumulated_score = 0
    for individual, score in zip(population, scores):
        accumulated_score += score
        if accumulated_score >= spin:
            return individual

In [171]:
print(roulette_wheel_selection(population, scores))

C


In [72]:
import random

def tournament_selection(population: list[str], scores: list[float], tournament_size:int) -> str:
    """Selects k unique participants, organizes tournament, the one with the highest score wins."""
    participants_index = random.sample(range(len(population)), tournament_size)
    champion_index = max(participants_index, key=lambda x: scores[x])
    return population[champion_index]

In [92]:
print(tournament_selection(population, scores, 2))

C


In [195]:
import random

def ranked_bases_selection(population: list[str], scores: list[float]) -> str:
    # sorting the indices by the fitness score
    sorted_indices = sorted(range(len(scores)), key=lambda x: scores[x])
    # it has the shape [(rank, index in population)]
    ranked_indices =  enumerate(sorted_indices, start=1)
    ranked_population = [(rank, population[population_index]) for rank, population_index in ranked_indices]
    # 1 + 2 + 3 + 4 + ... + n = n * (n+1) / 2
    total = len(population) * (len(population) + 1) // 2
    spin = random.uniform(0, total)
    accumulated = 0
    for rank, individual in ranked_population:
        accumulated += rank
        if accumulated >= spin:
            return individual

In [209]:
ranked_bases_selection(population, scores)

[(1, 'B'), (2, 'D'), (3, 'C'), (4, 'A')]
1.6910139910106903


'D'